# Ch.3 演習ノートブック：異常検知入門

| 項目 | 内容 |
|------|------|
| **使用データ** | 手書き数字データ（`load_digits()`）1,797枚・64特徴量 |
| **作るもの** | 異常スコアランキング ＋ 異常サンプルの可視化 |
| **実務での対応物** | 製造ラインの不良品検査レポート / センサー異常アラートの原型 |
| **時間の目安** | 演習 40分（問3まで完了で十分） |

## このノートブックの使い方

座学で学んだ「**正常の典型像から遠いものを異常とする**」考え方をコードで体験します。

1. 問1：「正常の基準（平均プロファイル）」を作る
2. 問2：「正常からの遠さ（異常スコア）」を計算して可視化する
3. 問3：「なぜ異常か」を人間の目で解釈する

> ✅ **問3まで完了すれば十分です。**

---
## 🤖 GitHub Copilot の使い方（このコースでの活用ルール）

### JupyterLab で Copilot を使う 2 つの方法

| 方法 | 手順 |
|------|------|
| **コメントから補完** | コードセルに `# 〇〇するコードを書いて` と書いて **Tab** キー |
| **チャットで質問** | 右サイドバーの Copilot Chat に日本語で質問する |

### このコースでの活用ルール

| タイミング | ルール |
|-----------|--------|
| 座学中 | **禁止** ― まず自分の頭で概念を理解する |
| 演習 問1・2 | **詰まったら使う** ― まず 3 分間は自力で考える |
| 演習 問3（考察） | コードは OK、**考察文は自分で書く** |
| 演習 問4（発展） | **自由に活用** ― プロンプトの工夫も練習 |

> 💡 **Copilot を使う前に「何をしたいか」を日本語で言えるようにしましょう。**  
> 「何をしたいか」が言えないと良いプロンプトが書けません。それがわかれば Copilot が助けてくれます。

---
## 🔧 準備

### ライブラリを読み込む

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import japanize_matplotlib
except ImportError:
    pass
from sklearn.datasets import load_digits

print("✅ ライブラリ読み込み完了")

### データを読み込む

In [ ]:
digits = load_digits()
X = digits.data    # 形状 (1797, 64) ← 1797枚 × 64ピクセル
y = digits.target  # 正解ラベル（0〜9の数字）

print(f"データの形: {X.shape}")
print(f"含まれる数字: {sorted(set(y))}")
print(f"各数字のサンプル数: 約 {len(X)//10} 件/数字")

### ランダムに数字の画像を表示して確認する

In [ ]:
# 10種類の数字をそれぞれ1枚ずつ表示する
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in zip(range(10), axes.flatten()):
    idx = np.where(y == digit)[0][0]   # その数字の最初のサンプル
    ax.imshow(X[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"数字「{digit}」")
    ax.axis("off")
plt.suptitle("手書き数字のサンプル（各1枚）", fontsize=13)
plt.tight_layout()
plt.show()

#### 実務との接続
この画像データと同じ考え方が実務で使われています：

| 実務のデータ | 類似点 |
|------------|--------|
| 製造ラインの部品画像 | 「正常な部品」の典型像と比較して不良品を検出 |
| 心電図データ | 「正常な波形」の平均から外れた波形を異常検知 |
| センサーの時系列データ | 「正常な数値パターン」から外れた時点をアラート |

---
## 📋 座学デモ再現：数字「0」の正常プロファイルを作る

### STEP 1：数字「0」のサンプルだけ取り出す

In [ ]:
# Ch.1 で学んだ「条件でデータを絞り込む」の応用
X_zeros = X[y == 0]
print(f"数字「0」のサンプル数: {len(X_zeros)} 件")

### STEP 2：平均プロファイルを計算する

`axis=0` は「各ピクセル位置で全サンプルを平均する」という意味です。

In [ ]:
mean_profile_0 = X_zeros.mean(axis=0)   # 各ピクセルの平均値
std_profile_0  = X_zeros.std(axis=0)    # 各ピクセルの標準偏差（ブレ具合）

print(f"平均プロファイルの形: {mean_profile_0.shape}  ← 64ピクセル分")

### STEP 3：平均プロファイルを画像として表示する

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))

# 平均プロファイル
axes[0].imshow(mean_profile_0.reshape(8, 8), cmap="gray_r")
axes[0].set_title("平均プロファイル\n（正常の基準）")
axes[0].axis("off")

# 標準偏差マップ
axes[1].imshow(std_profile_0.reshape(8, 8), cmap="Reds")
axes[1].set_title("標準偏差マップ\n（赤 = ブレが大きい）")
axes[1].axis("off")

# 実際のサンプル3枚を並べて比較
axes[2].imshow(X_zeros[0].reshape(8, 8), cmap="gray_r")
axes[2].set_title("実際のサンプル例\n（1枚目）")
axes[2].axis("off")

plt.suptitle("数字「0」の統計プロファイル", fontsize=13)
plt.tight_layout()
plt.show()

### STEP 4：異常スコアを計算する関数を定義する

In [ ]:
def calc_anomaly_score(sample, mean, std):
    """
    異常スコアの計算（Zスコアの平均）
    - 各ピクセルで「平均との差 ÷ 標準偏差」を計算
    - 64ピクセル分を平均したものが異常スコア
    - スコアが高い = 正常の基準から遠い = 異常っぽい
    """
    z = np.abs((sample - mean) / (std + 1e-8))   # 1e-8 はゼロ除算防止
    return z.mean()

print("✅ 異常スコア計算関数を定義しました")

### STEP 5：全サンプルの異常スコアを計算する

In [ ]:
scores_0 = np.array([calc_anomaly_score(x, mean_profile_0, std_profile_0)
                     for x in X_zeros])

print(f"異常スコアの統計：")
print(f"  最小値: {scores_0.min():.2f}  （最も正常らしいサンプル）")
print(f"  平均値: {scores_0.mean():.2f}")
print(f"  最大値: {scores_0.max():.2f}  （最も異常っぽいサンプル）")

### STEP 6：最も異常っぽい「0」トップ5を表示する

In [ ]:
top5_idx = np.argsort(scores_0)[::-1][:5]

fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
for i, idx in enumerate(top5_idx):
    axes[i].imshow(X_zeros[idx].reshape(8, 8), cmap="gray_r")
    axes[i].set_title(f"#{i+1}\nスコア:{scores_0[idx]:.2f}")
    axes[i].axis("off")
plt.suptitle("数字「0」の中で最も異常っぽいサンプル トップ5", fontsize=12)
plt.tight_layout()
plt.show()

---
## 問1：数字「1」の平均プロファイルを作る ★☆☆

**上の STEP 1〜3 を「0」→「1」に変えて実行します。**

### なぜやるのか
「0」と「1」ではプロファイルの形が大きく違うはずです。  
「各数字に固有の平均プロファイルが作れる」ことを確認するのが目的です。

### ヒント：変えるのは1か所だけ
```python
X_zeros = X[y == 0]   ← これを
X_ones  = X[y == 1]   ← これに変える
```

In [ ]:
# 数字「1」のサンプルを取り出す
X_ones = X[y == 1]
print(f"数字「1」のサンプル数: {len(X_ones)} 件")

In [ ]:
# 平均プロファイルと標準偏差を計算する
mean_profile_1 = X_ones.mean(axis=0)
std_profile_1  = X_ones.std(axis=0)

In [ ]:
# 「0」と「1」のプロファイルを並べて比較する
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

axes[0, 0].imshow(mean_profile_0.reshape(8, 8), cmap="gray_r")
axes[0, 0].set_title("「0」の平均プロファイル"); axes[0, 0].axis("off")

axes[0, 1].imshow(mean_profile_1.reshape(8, 8), cmap="gray_r")
axes[0, 1].set_title("「1」の平均プロファイル"); axes[0, 1].axis("off")

axes[1, 0].imshow(std_profile_0.reshape(8, 8), cmap="Reds")
axes[1, 0].set_title("「0」の標準偏差"); axes[1, 0].axis("off")

axes[1, 1].imshow(std_profile_1.reshape(8, 8), cmap="Reds")
axes[1, 1].set_title("「1」の標準偏差"); axes[1, 1].axis("off")

plt.suptitle("「0」と「1」のプロファイル比較", fontsize=13)
plt.tight_layout()
plt.show()

#### ✅ チェックポイント
- 「1」の平均プロファイルは縦線の形になっていますか？
- 「0」（輪っか）と「1」（縦線）でプロファイルの形が違いますか？
- 標準偏差マップの赤い部分（ブレが大きい場所）はどこですか？

---
## 問2：「最も変な1」トップ5を見つける ★★☆

**実務での対応：** センサーデータで「最も異常値が大きかった時刻トップ5」を特定する作業と同じです。

### 全サンプルの異常スコアを計算する

In [ ]:
# 上で定義した calc_anomaly_score 関数を使う
scores_1 = np.array([calc_anomaly_score(x, mean_profile_1, std_profile_1)
                     for x in X_ones])

print(f"「1」の異常スコア統計：")
print(f"  最小: {scores_1.min():.2f}  平均: {scores_1.mean():.2f}  最大: {scores_1.max():.2f}")

### スコアの分布をヒストグラムで確認する

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(scores_1, bins=20, color="#E86A2C", edgecolor="white")
plt.axvline(scores_1.mean(), color="navy", linestyle="--", label=f"平均 {scores_1.mean():.2f}")
plt.axvline(scores_1.mean() + 2*scores_1.std(), color="red",
            linestyle="--", label=f"平均+2σ {scores_1.mean()+2*scores_1.std():.2f}")
plt.xlabel("異常スコア")
plt.ylabel("件数")
plt.title("「1」の異常スコア分布")
plt.legend()
plt.tight_layout()
plt.show()
print(f"平均+2σ を超えるサンプル数: {(scores_1 > scores_1.mean() + 2*scores_1.std()).sum()} 件")

### トップ5の画像を表示する

In [ ]:
top5_idx_1 = np.argsort(scores_1)[::-1][:5]

fig, axes = plt.subplots(2, 5, figsize=(11, 5))

# 上段：異常っぽいサンプル
for i, idx in enumerate(top5_idx_1):
    axes[0, i].imshow(X_ones[idx].reshape(8, 8), cmap="gray_r")
    axes[0, i].set_title(f"異常#{i+1}\nスコア:{scores_1[idx]:.2f}")
    axes[0, i].axis("off")

# 下段：正常っぽいサンプル（スコアが低い5件）
bottom5_idx_1 = np.argsort(scores_1)[:5]
for i, idx in enumerate(bottom5_idx_1):
    axes[1, i].imshow(X_ones[idx].reshape(8, 8), cmap="gray_r")
    axes[1, i].set_title(f"正常#{i+1}\nスコア:{scores_1[idx]:.2f}")
    axes[1, i].axis("off")

plt.suptitle("上段：最も異常っぽい「1」  /  下段：最も正常っぽい「1"", fontsize=12)
plt.tight_layout()
plt.show()

#### ✅ チェックポイント
- 上段（異常）と下段（正常）で画像の見た目に違いがありますか？
- 上段のサンプルは確かに「変な1」に見えますか？

---
## 問3：なぜ異常と判定されたかを考察する ★★☆（最重要）

**実務での対応：** 工場の品質管理担当者に「なぜこの部品を不良と判定したか」を説明するレポートと同じです。

### 異常を「人間の目」で確認することが重要な理由

機械が「変だ」と言っても、それが **本当に問題なのか** を判断するのは人間です。  
「スコアが高い → 画像を目視 → 確かに変 → 原因を推定 → 対策を提案」  
この流れが実務でのデータサイエンティストの仕事です。

### 考察のヒント
- 形が通常の「1」と違う（大きく傾いている・曲がっている）
- 書いた場所が画像の端に寄っている
- 線が途切れている・余計な点が入っている
- 薄すぎる・太すぎる

### 考察欄

**異常1位のサンプルを見て気づいたこと：**

> （記入してください）

**なぜ異常スコアが高いと思いますか？**

> （例：縦線が左に大きく傾いており、平均プロファイルの縦線の位置と大きくずれているから）

**実務に当てはめると：**

> （例：製造ラインなら「特定の方向に傾いた部品が多発 → 金型のズレが原因かもしれない」のようなアクションにつながる）

---
> 💡 この考察を Copilot に書かせないでください。  
> 「グラフを見て言語化する能力」が実務でのあなたの価値です。

---
## 問4（発展）：IsolationForest で機械学習ベースの異常検知 ★★★

### IsolationForest とは
- 「孤立しやすいデータ = 異常」という考え方で検知するアルゴリズム
- 統計的な平均・標準偏差ではなく、決定木を使った機械学習の手法
- `contamination` パラメータで「全体の何%を異常と判定するか」を設定できる

**実務での使われ方：** クレジットカードの不正検知・ECサイトのボット検出など

In [ ]:
from sklearn.ensemble import IsolationForest

# IsolationForest でモデルを作る
iso_model = IsolationForest(contamination=0.05, random_state=42)
iso_model.fit(X_ones)

# 予測（-1 = 異常、+1 = 正常）
pred_iso = iso_model.predict(X_ones)
n_anomaly = (pred_iso == -1).sum()
print(f"IsolationForest が異常と判定した件数: {n_anomaly} 件  （全体の {n_anomaly/len(X_ones):.1%}）")

In [ ]:
# 異常と判定されたサンプルを表示する
anomaly_idx_iso = np.where(pred_iso == -1)[0]
show_n = min(10, len(anomaly_idx_iso))

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, idx in enumerate(anomaly_idx_iso[:show_n]):
    row, col = divmod(i, 5)
    axes[row, col].imshow(X_ones[idx].reshape(8, 8), cmap="gray_r")
    axes[row, col].set_title(f"idx={idx}")
    axes[row, col].axis("off")

plt.suptitle("IsolationForest が異常と判定した「1」サンプル", fontsize=12)
plt.tight_layout()
plt.show()

> 🤖 **Copilot チャレンジ**
>
> 数字「0」以外の数字（例：「8」や「9」）でも異常検知を試してみましょう：
> ```
> # 数字「9」について平均プロファイルを作り、異常スコアを計算して
> # 最も異常っぽいトップ5を表示するコードを書いて
> ```
> どの数字が「バリエーションが多い（異常が多い）」と思いますか？